# Problème du Voyageur de Commerce
## Algorithme Génétique (Version sans Classes)

**Objectif:** Minimiser F = distance + coût

In [ ]:
import csv
import random
import os

## 1. Paramètres de l'Algorithme

In [ ]:
# Configuration (simple dictionnaire)
PARAMS = {
    'taille_population': 300,
    'nb_generations': 500,
    'taux_croisement': 0.70,
    'taux_mutation': 0.15,
    'taille_tournoi': 5
}

## 2. Chargement des Données

In [ ]:
def lire_matrice_csv(chemin_fichier):
    """
    Lit une matrice depuis un fichier CSV.
    Retourne: (matrice, liste_villes)
    """
    matrice = []
    villes = []
    
    with open(chemin_fichier, 'r', encoding='utf-8') as fichier:
        lecteur = csv.reader(fichier)
        
        # Première ligne = en-tête avec noms des villes
        entete = next(lecteur)
        villes = entete[1:]  # Ignorer la première colonne
        
        # Lire les lignes de données
        for ligne in lecteur:
            if ligne:
                valeurs = [int(v) for v in ligne[1:]]  # Ignorer première colonne
                matrice.append(valeurs)
    
    return matrice, villes


def charger_donnees(fichier_distances, fichier_couts):
    """
    Charge les données du problème TSP.
    Retourne un dictionnaire avec toutes les données.
    """
    distances, villes = lire_matrice_csv(fichier_distances)
    couts, _ = lire_matrice_csv(fichier_couts)
    
    donnees = {
        'villes': villes,
        'distances': distances,
        'couts': couts
    }
    
    print(f"Données chargées: {len(villes)} villes")
    return donnees

In [ ]:
# Charger les données
DONNEES = charger_donnees("distances.csv", "cities.csv")
print(f"Villes: {DONNEES['villes']}")

## 3. Fonctions pour Calculer Distance et Coût

In [ ]:
def obtenir_distance(ville_a, ville_b, donnees):
    """Retourne la distance entre deux villes."""
    i = donnees['villes'].index(ville_a)
    j = donnees['villes'].index(ville_b)
    return donnees['distances'][i][j]


def obtenir_cout(ville_a, ville_b, donnees):
    """Retourne le coût entre deux villes."""
    i = donnees['villes'].index(ville_a)
    j = donnees['villes'].index(ville_b)
    return donnees['couts'][i][j]

## 4. Représentation d'une Solution

Une solution est un **dictionnaire** avec:
- `parcours`: liste des villes dans l'ordre de visite
- `distance`: distance totale (f1)
- `cout`: coût total (f2)
- `score`: f1 + f2

In [ ]:
def creer_solution_vide():
    """Crée une solution vide."""
    return {
        'parcours': [],
        'distance': 0,
        'cout': 0,
        'score': 0
    }


def creer_solution_aleatoire(donnees):
    """Crée une solution avec un parcours aléatoire."""
    parcours = donnees['villes'].copy()
    random.shuffle(parcours)
    
    return {
        'parcours': parcours,
        'distance': 0,
        'cout': 0,
        'score': 0
    }


def copier_solution(solution):
    """Crée une copie d'une solution."""
    return {
        'parcours': solution['parcours'].copy(),
        'distance': solution['distance'],
        'cout': solution['cout'],
        'score': solution['score']
    }

## 5. Évaluation d'une Solution

In [ ]:
def evaluer_solution(solution, donnees):
    """
    Calcule la distance, le coût et le score d'une solution.
    Modifie la solution en place.
    """
    parcours = solution['parcours']
    distance_totale = 0
    cout_total = 0
    
    # Calculer pour chaque étape
    for i in range(len(parcours) - 1):
        ville_depart = parcours[i]
        ville_arrivee = parcours[i + 1]
        distance_totale += obtenir_distance(ville_depart, ville_arrivee, donnees)
        cout_total += obtenir_cout(ville_depart, ville_arrivee, donnees)
    
    # Ajouter le retour à la ville de départ
    derniere_ville = parcours[-1]
    premiere_ville = parcours[0]
    distance_totale += obtenir_distance(derniere_ville, premiere_ville, donnees)
    cout_total += obtenir_cout(derniere_ville, premiere_ville, donnees)
    
    # Mettre à jour la solution
    solution['distance'] = distance_totale
    solution['cout'] = cout_total
    solution['score'] = distance_totale + cout_total

## 6. Opérateurs Génétiques

In [ ]:
def croisement_deux_points(parent1, parent2, donnees):
    """
    Croisement à deux points.
    Retourne une nouvelle solution (enfant).
    """
    parcours1 = parent1['parcours']
    parcours2 = parent2['parcours']
    taille = len(parcours1)
    
    # Choisir deux points de coupure
    point1 = random.randint(1, taille - 2)
    point2 = random.randint(point1 + 1, taille - 1)
    
    # Commencer avec le parcours du parent1
    nouveau_parcours = parcours1.copy()
    
    # Copier le segment du parent2 entre les deux points
    for i in range(point1, point2):
        nouveau_parcours[i] = parcours2[i]
    
    # Trouver les doublons et les villes manquantes
    villes_vues = set()
    positions_doublons = []
    
    for position, ville in enumerate(nouveau_parcours):
        if ville in villes_vues:
            positions_doublons.append(position)
        else:
            villes_vues.add(ville)
    
    # Trouver les villes manquantes
    toutes_villes = set(donnees['villes'])
    villes_manquantes = list(toutes_villes - villes_vues)
    random.shuffle(villes_manquantes)
    
    # Remplacer les doublons par les villes manquantes
    for i, position in enumerate(positions_doublons):
        nouveau_parcours[position] = villes_manquantes[i]
    
    # Créer la nouvelle solution
    enfant = creer_solution_vide()
    enfant['parcours'] = nouveau_parcours
    return enfant


def mutation_echange(solution, taux_mutation):
    """
    Mutation par échange de deux villes.
    Modifie la solution en place.
    """
    if random.random() < taux_mutation:
        parcours = solution['parcours']
        taille = len(parcours)
        
        if taille >= 2:
            # Choisir deux positions différentes
            i = random.randint(0, taille - 1)
            j = random.randint(0, taille - 1)
            while j == i:
                j = random.randint(0, taille - 1)
            
            # Échanger les deux villes
            parcours[i], parcours[j] = parcours[j], parcours[i]

## 7. Sélection par Tournoi

In [ ]:
def selection_tournoi(population, taille_tournoi):
    """
    Sélectionne une solution par tournoi.
    Retourne la meilleure solution parmi un échantillon aléatoire.
    """
    # Choisir des participants aléatoires
    participants = random.sample(population, taille_tournoi)
    
    # Trouver le meilleur (score le plus bas)
    meilleur = participants[0]
    for solution in participants[1:]:
        if solution['score'] < meilleur['score']:
            meilleur = solution
    
    return meilleur

## 8. Algorithme Génétique Principal

In [ ]:
def creer_population_initiale(taille, donnees):
    """Crée une population de solutions aléatoires."""
    population = []
    for _ in range(taille):
        solution = creer_solution_aleatoire(donnees)
        population.append(solution)
    return population


def evaluer_population(population, donnees):
    """Évalue toutes les solutions de la population."""
    for solution in population:
        evaluer_solution(solution, donnees)


def trier_population(population):
    """Trie la population par score croissant (meilleur en premier)."""
    population.sort(key=lambda s: s['score'])


def creer_nouvelle_generation(population, donnees, params):
    """Crée la génération suivante."""
    nouvelle_population = []
    
    # Élitisme: garder le meilleur
    meilleur = copier_solution(population[0])
    nouvelle_population.append(meilleur)
    
    # Remplir le reste de la population
    while len(nouvelle_population) < params['taille_population']:
        
        if random.random() < params['taux_croisement']:
            # Croisement
            parent1 = selection_tournoi(population, params['taille_tournoi'])
            parent2 = selection_tournoi(population, params['taille_tournoi'])
            enfant = croisement_deux_points(parent1, parent2, donnees)
        else:
            # Copie simple
            parent = selection_tournoi(population, params['taille_tournoi'])
            enfant = copier_solution(parent)
        
        # Mutation
        mutation_echange(enfant, params['taux_mutation'])
        
        nouvelle_population.append(enfant)
    
    return nouvelle_population

In [ ]:
def executer_algorithme_genetique(donnees, params):
    """
    Exécute l'algorithme génétique complet.
    Retourne la meilleure solution trouvée.
    """
    print(f"Démarrage - Population: {params['taille_population']}, Générations: {params['nb_generations']}")
    print("=" * 70)
    
    # Créer la population initiale
    population = creer_population_initiale(params['taille_population'], donnees)
    
    # Variables pour suivre le meilleur global
    meilleur_global = None
    generation_meilleur = 0
    
    # Boucle principale
    for generation in range(params['nb_generations']):
        
        # Évaluer et trier
        evaluer_population(population, donnees)
        trier_population(population)
        
        # Vérifier si on a trouvé un meilleur
        champion = population[0]
        
        if meilleur_global is None or champion['score'] < meilleur_global['score']:
            meilleur_global = copier_solution(champion)
            generation_meilleur = generation
            print(f"Gén {generation:4d}: F={champion['score']} (dist={champion['distance']}, coût={champion['cout']})")
        
        # Créer la génération suivante
        population = creer_nouvelle_generation(population, donnees, params)
    
    print("=" * 70)
    print(f"\nMeilleure solution trouvée à la génération {generation_meilleur}")
    
    return meilleur_global

## 9. Fonctions d'Affichage et Sauvegarde

In [ ]:
def afficher_solution(solution):
    """Affiche une solution de manière lisible."""
    parcours = solution['parcours']
    trajet = " → ".join(parcours + [parcours[0]])
    print(f"Score F = {solution['score']} (Distance={solution['distance']}, Coût={solution['cout']})")
    print(f"Parcours: {trajet}")


def solution_vers_chaine(solution):
    """Convertit le parcours en chaîne CSV."""
    parcours = solution['parcours']
    return ",".join(parcours + [parcours[0]])

In [ ]:
def charger_solutions_xml(fichier):
    """
    Charge les solutions depuis le fichier XML.
    Retourne: (meilleur_score, liste_parcours)
    """
    if not os.path.exists(fichier):
        return -1, []
    
    try:
        with open(fichier, 'r', encoding='utf-8') as f:
            contenu = f.read()
        
        # Chercher le score
        import re
        match_score = re.search(r'<BestFitness>(\d+)</BestFitness>', contenu)
        if not match_score:
            match_score = re.search(r'<Fitness>(\d+)</Fitness>', contenu)
        
        if not match_score:
            return -1, []
        
        score = int(match_score.group(1))
        
        # Chercher tous les parcours
        parcours_list = re.findall(r'<Tour>([^<]+)</Tour>', contenu)
        
        return score, parcours_list
    
    except Exception as e:
        print(f"Erreur: {e}")
        return -1, []


def sauvegarder_solutions_xml(fichier, score, distance, cout, liste_parcours):
    """Sauvegarde les solutions dans un fichier XML."""
    
    # Construire le contenu XML
    lignes = [
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<TSP_Solutions>',
        f'    <BestFitness>{score}</BestFitness>',
        f'    <TotalSolutions>{len(liste_parcours)}</TotalSolutions>',
        ''
    ]
    
    for i, parcours in enumerate(liste_parcours, 1):
        lignes.append(f'    <Solution id="{i}">')
        lignes.append(f'        <Tour>{parcours}</Tour>')
        lignes.append(f'        <Distance>{distance}</Distance>')
        lignes.append(f'        <Cost>{cout}</Cost>')
        lignes.append('    </Solution>')
        lignes.append('')
    
    lignes.append('</TSP_Solutions>')
    
    # Écrire dans le fichier
    with open(fichier, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lignes))
    
    print(f"Solution sauvegardée dans {fichier}")

## 10. Exécution Principale

In [ ]:
# Charger les solutions précédentes
FICHIER_XML = "solution.xml"
meilleur_precedent, parcours_existants = charger_solutions_xml(FICHIER_XML)

if meilleur_precedent > 0:
    print(f"Meilleur score précédent: {meilleur_precedent}")
    print(f"Nombre de solutions: {len(parcours_existants)}")
    print()

# Exécuter l'algorithme
resultat = executer_algorithme_genetique(DONNEES, PARAMS)

# Afficher le résultat
print()
afficher_solution(resultat)

## 11. Sauvegarde du Résultat

In [ ]:
nouveau_parcours = solution_vers_chaine(resultat)

print("=" * 70)

if meilleur_precedent < 0:
    # Première solution
    sauvegarder_solutions_xml(FICHIER_XML, resultat['score'], resultat['distance'], resultat['cout'], [nouveau_parcours])
    print("PREMIÈRE SOLUTION ENREGISTRÉE")

elif resultat['score'] < meilleur_precedent:
    # Nouveau record
    sauvegarder_solutions_xml(FICHIER_XML, resultat['score'], resultat['distance'], resultat['cout'], [nouveau_parcours])
    print(f"NOUVEAU RECORD! ({meilleur_precedent} → {resultat['score']})")

elif resultat['score'] == meilleur_precedent:
    # Même score
    if nouveau_parcours not in parcours_existants:
        parcours_existants.append(nouveau_parcours)
        sauvegarder_solutions_xml(FICHIER_XML, resultat['score'], resultat['distance'], resultat['cout'], parcours_existants)
        print(f"NOUVEAU PARCOURS DÉCOUVERT (même score: {resultat['score']})")
        print(f"Total: {len(parcours_existants)} parcours")
    else:
        print(f"PARCOURS DÉJÀ CONNU (score: {resultat['score']})")

else:
    # Moins bon
    print(f"PAS D'AMÉLIORATION (actuel: {resultat['score']}, meilleur: {meilleur_precedent})")

print("=" * 70)

## 12. Afficher Toutes les Solutions

In [ ]:
# Recharger et afficher
score_final, tous_parcours = charger_solutions_xml(FICHIER_XML)

print(f"Meilleur score (F): {score_final}")
print(f"Nombre de parcours: {len(tous_parcours)}")
print()

for i, parcours in enumerate(tous_parcours, 1):
    print(f"Parcours {i}: {parcours}")

## 13. Recherche Intensive (Optionnel)

In [ ]:
def recherche_intensive(nb_executions):
    """
    Lance plusieurs exécutions pour trouver plus de solutions.
    """
    print(f"Lancement de {nb_executions} exécutions...")
    
    for i in range(nb_executions):
        print(f"\n{'='*70}")
        print(f"EXÉCUTION {i+1}/{nb_executions}")
        print(f"{'='*70}")
        
        # Charger l'état actuel
        meilleur_actuel, parcours_actuels = charger_solutions_xml(FICHIER_XML)
        
        # Exécuter l'algorithme
        resultat = executer_algorithme_genetique(DONNEES, PARAMS)
        nouveau_parcours = solution_vers_chaine(resultat)
        
        # Traiter le résultat
        if meilleur_actuel < 0 or resultat['score'] < meilleur_actuel:
            sauvegarder_solutions_xml(FICHIER_XML, resultat['score'], resultat['distance'], resultat['cout'], [nouveau_parcours])
            print(f"NOUVEAU RECORD: {resultat['score']}")
        elif resultat['score'] == meilleur_actuel and nouveau_parcours not in parcours_actuels:
            parcours_actuels.append(nouveau_parcours)
            sauvegarder_solutions_xml(FICHIER_XML, resultat['score'], resultat['distance'], resultat['cout'], parcours_actuels)
            print(f"NOUVEAU PARCOURS (total: {len(parcours_actuels)})")
    
    # Résumé final
    score_final, tous_parcours = charger_solutions_xml(FICHIER_XML)
    print(f"\n{'#'*70}")
    print(f"RÉSUMÉ: Score={score_final}, Parcours trouvés={len(tous_parcours)}")
    print(f"{'#'*70}")


# Décommenter pour lancer:
# recherche_intensive(10)